In [ ]:
import sys
import pandas as pd
from datetime import datetime, timedelta

# Tell the notebook to look in the main project folder
sys.path.append('..')
from src.data_loader import DataLoader
from src.models import XGBoostPoissonRegressor

# Set pandas display options for cleaner tables
pd.set_option('display.max_columns', None)

In [ ]:
# 1. Dynamic 2026 Season Dates
start_date = "2026-03-26"
current_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
end_of_season = "2026-09-27"

print(f"Loading multi-year Statcast data up to {current_date}...")

# 2. Instantiate Data Loader and pull the stabilized matrix
loader = DataLoader(start_dt=start_date, end_dt=current_date)
df = loader.get_clean_hitter_matrix(min_batted_balls=50)

print(f"\nData loaded successfully! Matrix shape: {df.shape}")

In [ ]:
# 1. Define the multi-year feature space
feature_cols = [
    'xwOBA', 'avg_exit_velocity', 'max_exit_velocity', 'avg_launch_angle', 'hard_hit_rate',
    'xwOBA_2025', 'hr_2025', 'xwOBA_2024', 'hr_2024'
]

X = df[feature_cols].values
y = df['home_runs'].values

# 2. Train the Model
print("Training Multi-Year XGBoost Engine...")
model = XGBoostPoissonRegressor()
model.fit(X, y)

# 3. Pacing Calculus
days_played = (datetime.strptime(current_date, "%Y-%m-%d") - datetime.strptime(start_date, "%Y-%m-%d")).days
total_season_days = (datetime.strptime(end_of_season, "%Y-%m-%d") - datetime.strptime(start_date, "%Y-%m-%d")).days
days_remaining = total_season_days - days_played

ros_multiplier = days_remaining / days_played
season_remaining_pct = days_remaining / total_season_days

# 4. Calculate Reliability & Blend
raw_base_projection = model.predict(X)
raw_projected_ros_hrs = raw_base_projection * ros_multiplier

baseline_ros_hrs = df['hr_2025'] * season_remaining_pct
stabilization_constant = 100
reliability_weight = df['total_batted_balls'] / (df['total_batted_balls'] + stabilization_constant)

df['projected_ros_hrs'] = (reliability_weight * raw_projected_ros_hrs) + ((1 - reliability_weight) * baseline_ros_hrs)
df['projected_total_hrs'] = df['home_runs'] + df['projected_ros_hrs']

print("Projections calculated and stabilized!")

In [ ]:
# Sort by End of Year Total
top_hitters = df.sort_values('projected_total_hrs', ascending=False)

display_cols = [
    'batter_name', 'total_batted_balls', 'max_exit_velocity', 'hard_hit_rate', 
    'home_runs', 'projected_ros_hrs', 'projected_total_hrs'
]

# Display the top 15
top_hitters[display_cols].head(15).style.format({
    'max_exit_velocity': '{:.1f} mph',
    'hard_hit_rate': '{:.1%}',
    'projected_ros_hrs': '+{:.1f}',
    'projected_total_hrs': '{:.1f}'
}).background_gradient(subset=['projected_total_hrs'], cmap='Greens')